---
format:
    html: default
    ipynb: default
jupyter: python3
---


## Explaining CNNs


In this assignment you implement two explainer algorithms for Convolutional Neural Networks (CNNs) and use them to inspect a model you have already trained. An explainer answers a question that accuracy alone cannot: which parts of an input drove this particular prediction? For an image classifier that answer takes the form of a saliency map, and a trustworthy map should highlight the animal rather than the background, a watermark, or some dataset artifact the model latched onto.

We invite you to watch the following video, which sets the context and walks through the tools you can use. Click the thumbnail to open it on YouTube.

[![Watch the assignment introduction on YouTube](https://img.youtube.com/vi/Am2EF9CLu-g/hqdefault.jpg)](https://www.youtube.com/watch?v=Am2EF9CLu-g)

In the [cats vs dogs](https://huggingface.co/datasets/pantelism/cats-vs-dogs) [classification task](https://aegean.ai/aiml-common/lectures/cnn/cnn-example-architectures/using_convnets_with_small_datasets) you trained a model that, with the help of data augmentation, reached a useful level of accuracy without overfitting. That trained model is your subject here. You do not retrain it or change its architecture; you attach explainers to it and interpret what they reveal about how it decides.

### The two methods you will implement

**Integrated gradients** ([Sundararajan et al., 2017](https://arxiv.org/abs/1703.01365)) attributes a prediction to individual input pixels. It picks a baseline image (commonly an all-black image, which the model should find uninformative) and integrates the gradient of the class score along the straight-line path from that baseline to the actual input. The result is a per-pixel attribution with two properties that plain input gradients lack: completeness, meaning the attributions sum to the difference in model output between the input and the baseline, and robustness to saturated activations, where a plain gradient would read close to zero even though the feature mattered.

**Grad-CAM** ([Selvaraju et al., 2016](https://arxiv.org/abs/1610.02391)) produces a coarse, class-discriminative heatmap. It takes the feature maps of a convolutional layer, usually the last one, and weights each map by the gradient of the class score flowing into it. Because it works at the resolution of a deep feature map rather than the raw pixels, it localizes the region the network used instead of scoring every pixel.

The two methods sit at opposite ends of a resolution trade-off: integrated gradients is fine-grained and pixel-level, Grad-CAM is coarse and region-level. Running both on the same image and noting where they agree, and where they do not, is the heart of this assignment.

We strongly advise PyTorch with [Captum](https://captum.ai/tutorials/) unless you already have Keras/TF expertise, because both methods ship as ready implementations there (`IntegratedGradients` and `LayerGradCam`). You are free to use the high-level APIs of the framework of your choice.

### What to submit

For each method, in the cells below:

- A markdown explanation written so that anyone who understands how a CNN works can follow it. Cover the intuition, the role of the baseline (integrated gradients) or the target layer (Grad-CAM), and one limitation of the method.
- Working code that runs the explainer on at least three correctly classified images and at least one image the model got wrong.
- The resulting maps overlaid on the input images, each with a one-line caption stating what the map suggests the model attended to.

### How your work is evaluated

- Correctness: the right baseline, the right target layer, and gradients taken with respect to the predicted class rather than a fixed label.
- Visualization quality: maps are overlaid on the inputs, readable, and labeled.
- Depth of interpretation: noting that a map "looks reasonable" is not enough. Point to where the two methods agree, where they disagree, and what the misclassified example tells you about what the model actually learned.

# CNN Explainers: Integrated Gradients and Grad-CAM

**Name:** Jacob Moawad

This notebook uses the trained cats-versus-dogs CNN from the previous assignment. It does not retrain or change the model. It loads the saved weights, finds three correctly classified test images and one misclassified test image, and explains each prediction with Integrated Gradients and Grad-CAM.

## 1. Setup

The previous training notebook saves the regularized model as `cats_and_dogs_small.pth`. Run the next cells in the same folder as that file. In Google Colab, the notebook will ask you to upload the file if it is not already present.

In [ ]:
%pip install -q captum datasets

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from datasets import load_dataset
from captum.attr import IntegratedGradients, LayerGradCam, LayerAttribution

SEED = 42
IMG_SIZE = 150
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", DEVICE)

## 2. Load the dataset and trained model

The dataset uses label `0` for cat and label `1` for dog. The same preprocessing from training is used here: resize to $150 \times 150$, convert to a tensor, and normalize each channel with mean 0.5 and standard deviation 0.5.

In [ ]:
ds_dict = load_dataset("pantelism/cats-vs-dogs")
label_names = ds_dict["test"].features["label"].names

basic_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

class CatsDogsDataset(Dataset):
    def __init__(self, hf_dataset, transform):
        self.data = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        image = sample["image"].convert("RGB")
        label = int(sample["label"])
        return self.transform(image), label, idx

test_ds = CatsDogsDataset(ds_dict["test"], basic_tf)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Test images:", len(test_ds))
print("Labels:", label_names)

In [ ]:
class SmallConvNet(nn.Module):
    def __init__(self, dropout=True):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 128, 3), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5) if dropout else nn.Identity(),
            nn.Linear(128 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x).squeeze(1)

MODEL_PATH = Path("cats_and_dogs_small.pth")

if not MODEL_PATH.exists():
    try:
        from google.colab import files
        print("Upload cats_and_dogs_small.pth from the previous assignment.")
        uploaded = files.upload()
        if "cats_and_dogs_small.pth" not in uploaded:
            raise FileNotFoundError("The uploaded file must be named cats_and_dogs_small.pth")
    except ImportError as exc:
        raise FileNotFoundError(
            "Place cats_and_dogs_small.pth in the same folder as this notebook."
        ) from exc

model = SmallConvNet(dropout=True).to(DEVICE)

try:
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
except TypeError:
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)

if isinstance(checkpoint, nn.Module):
    model = checkpoint.to(DEVICE)
else:
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        checkpoint = checkpoint["state_dict"]
    checkpoint = {key.replace("module.", "", 1): value for key, value in checkpoint.items()}
    model.load_state_dict(checkpoint)

model.eval()
print("Loaded:", MODEL_PATH)

## 3. Select the examples

I evaluate the whole test set first. I select at least one correctly classified cat, one correctly classified dog, one additional correct example, and the most confident wrong prediction. Choosing a confident mistake is useful because it can reveal what misleading evidence the model relied on.

In [ ]:
all_probs = np.zeros(len(test_ds), dtype=np.float32)
all_labels = np.zeros(len(test_ds), dtype=np.int64)

with torch.no_grad():
    for images, labels, indices in test_loader:
        logits = model(images.to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()
        indices_np = indices.numpy()
        all_probs[indices_np] = probs
        all_labels[indices_np] = labels.numpy()

all_preds = (all_probs >= 0.5).astype(int)
all_confidence = np.where(all_preds == 1, all_probs, 1.0 - all_probs)
accuracy = (all_preds == all_labels).mean()
print(f"Test accuracy: {accuracy:.3f}")

correct_cat = np.where((all_preds == all_labels) & (all_labels == 0))[0]
correct_dog = np.where((all_preds == all_labels) & (all_labels == 1))[0]
correct_all = np.where(all_preds == all_labels)[0]
wrong_all = np.where(all_preds != all_labels)[0]

if len(correct_all) < 3:
    raise RuntimeError("The model produced fewer than three correct test predictions.")
if len(wrong_all) == 0:
    raise RuntimeError("The model produced no wrong prediction, so a required example is missing.")

selected_correct = []
if len(correct_cat) > 0:
    selected_correct.append(int(correct_cat[np.argmax(all_confidence[correct_cat])]))
if len(correct_dog) > 0:
    selected_correct.append(int(correct_dog[np.argmax(all_confidence[correct_dog])]))

for idx in correct_all[np.argsort(all_confidence[correct_all])[::-1]]:
    idx = int(idx)
    if idx not in selected_correct:
        selected_correct.append(idx)
    if len(selected_correct) == 3:
        break

wrong_idx = int(wrong_all[np.argmax(all_confidence[wrong_all])])
selected_indices = selected_correct[:3] + [wrong_idx]

selection_table = pd.DataFrame({
    "test_index": selected_indices,
    "true_label": [label_names[all_labels[i]] for i in selected_indices],
    "prediction": [label_names[all_preds[i]] for i in selected_indices],
    "confidence": [all_confidence[i] for i in selected_indices],
    "result": ["correct" if all_preds[i] == all_labels[i] else "wrong" for i in selected_indices],
})
display(selection_table)

# Integrated Gradients

## Explanation

Integrated Gradients starts from a baseline image and moves toward the real image using many small steps. At every step it calculates the gradient of the predicted-class score with respect to every input pixel. It averages those gradients and multiplies them by the difference between the image and the baseline. Pixels with larger attribution values had a larger effect on the prediction.

I use an all-black image as the baseline because it should contain little useful cat-or-dog information. The input tensors were normalized using mean 0.5 and standard deviation 0.5, so a black pixel is represented by $-1$, not 0. Therefore, the correct black baseline in the normalized input space is a tensor filled with $-1$.

For this binary classifier, the model directly outputs a dog logit. For a predicted dog, I explain the positive logit. For a predicted cat, I explain the negative of that logit. This makes the gradient correspond to the class the model actually predicted rather than always explaining the dog class.

A limitation is that the result depends on the baseline. A different baseline can change the attribution map. Integrated Gradients is also detailed but sometimes noisy, and an absolute-value visualization shows importance strength while hiding whether a pixel supported or opposed the class.

# Grad-CAM

## Explanation

Grad-CAM explains a prediction using the feature maps from a convolutional layer. It calculates the gradient of the predicted-class score with respect to every channel in the selected layer. The spatial average of each channel's gradient becomes that channel's weight. The weighted feature maps are added together, passed through ReLU, and enlarged to the input image size.

I use `model.features[9]`, the last `Conv2d` layer. A deep convolutional layer contains higher-level visual patterns and still keeps spatial information. Using the final convolutional layer therefore gives a class-specific region map before the network flattens the features for classification.

A limitation is that Grad-CAM is coarse. The last convolutional feature maps have a much lower resolution than the original image, so the final heatmap can identify a general region but not exact pixels. The result can also change when a different target layer is selected.

## 4. Explainer implementation

Both explainers below receive a wrapper that returns the score of the **predicted class**. The model itself and its architecture are not changed.

In [ ]:
class PredictedClassScore(nn.Module):
    # For dog, use the dog logit. For cat, use its negative.
    def __init__(self, base_model, class_index):
        super().__init__()
        self.base_model = base_model
        self.class_index = int(class_index)

    def forward(self, x):
        logit = self.base_model(x)
        score = logit if self.class_index == 1 else -logit
        return score.unsqueeze(1)


def normalize_heatmap(heatmap):
    heatmap = np.asarray(heatmap, dtype=np.float32)
    heatmap = heatmap - heatmap.min()
    maximum = heatmap.max()
    if maximum > 0:
        heatmap = heatmap / maximum
    return heatmap


def tensor_to_image(x):
    image = (x.detach().cpu().squeeze(0) * 0.5 + 0.5).clamp(0, 1)
    return image.permute(1, 2, 0).numpy()


def make_overlay(image, heatmap, cmap_name="inferno"):
    colors = plt.get_cmap(cmap_name)(heatmap)[..., :3]
    alpha = (0.58 * heatmap)[..., None]
    return np.clip((1 - alpha) * image + alpha * colors, 0, 1)


def heatmap_location(heatmap):
    total = float(heatmap.sum())
    if total <= 1e-12:
        return "around the center"
    h, w = heatmap.shape
    yy, xx = np.mgrid[0:h, 0:w]
    y = float((yy * heatmap).sum() / total) / max(h - 1, 1)
    x = float((xx * heatmap).sum() / total) / max(w - 1, 1)
    vertical = "upper" if y < 0.38 else "lower" if y > 0.62 else "middle"
    horizontal = "left" if x < 0.38 else "right" if x > 0.62 else "center"
    if vertical == "middle" and horizontal == "center":
        return "around the center"
    return f"in the {vertical}-{horizontal} area"


def center_attention_ratio(heatmap):
    h, w = heatmap.shape
    y1, y2 = int(0.20 * h), int(0.80 * h)
    x1, x2 = int(0.20 * w), int(0.80 * w)
    total = float(heatmap.sum()) + 1e-12
    return float(heatmap[y1:y2, x1:x2].sum() / total)


def attention_description(heatmap):
    ratio = center_attention_ratio(heatmap)
    if ratio >= 0.58:
        return "mostly on the central subject area"
    if ratio <= 0.40:
        return "more on the outer or background areas"
    return "between the central subject and its surroundings"


def top_region_iou(a, b, quantile=0.80):
    a_mask = a >= np.quantile(a, quantile)
    b_mask = b >= np.quantile(b, quantile)
    union = np.logical_or(a_mask, b_mask).sum()
    if union == 0:
        return 0.0
    return float(np.logical_and(a_mask, b_mask).sum() / union)


def explain_one_image(test_index, n_steps=64):
    sample = ds_dict["test"][int(test_index)]
    pil_image = sample["image"].convert("RGB")
    true_label = int(sample["label"])
    x = basic_tf(pil_image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        dog_probability = torch.sigmoid(model(x))[0].item()

    predicted_class = int(dog_probability >= 0.5)
    confidence = dog_probability if predicted_class == 1 else 1.0 - dog_probability
    score_model = PredictedClassScore(model, predicted_class).to(DEVICE).eval()

    # Black pixels are -1 after normalization.
    baseline = torch.full_like(x, -1.0)
    ig = IntegratedGradients(score_model)
    ig_attr, delta = ig.attribute(
        x,
        baselines=baseline,
        target=0,
        n_steps=n_steps,
        return_convergence_delta=True,
    )
    ig_heatmap = ig_attr[0].detach().abs().sum(dim=0).cpu().numpy()
    ig_heatmap = normalize_heatmap(ig_heatmap)

    # The last convolutional layer is features[9].
    target_layer = model.features[9]
    grad_cam = LayerGradCam(score_model, target_layer)
    cam_attr = grad_cam.attribute(x, target=0)
    cam_attr = LayerAttribution.interpolate(
        cam_attr,
        (IMG_SIZE, IMG_SIZE),
        interpolate_mode="bilinear",
    )
    cam_heatmap = torch.relu(cam_attr[0, 0]).detach().cpu().numpy()
    cam_heatmap = normalize_heatmap(cam_heatmap)

    image = tensor_to_image(x)
    return {
        "index": int(test_index),
        "image": image,
        "true_class": true_label,
        "predicted_class": predicted_class,
        "confidence": confidence,
        "correct": predicted_class == true_label,
        "ig_heatmap": ig_heatmap,
        "cam_heatmap": cam_heatmap,
        "ig_overlay": make_overlay(image, ig_heatmap),
        "cam_overlay": make_overlay(image, cam_heatmap),
        "ig_delta": float(delta.detach().cpu().flatten()[0]),
        "overlap": top_region_iou(ig_heatmap, cam_heatmap),
        "ig_center_ratio": center_attention_ratio(ig_heatmap),
        "cam_center_ratio": center_attention_ratio(cam_heatmap),
    }

## 5. Generate and display the explanations

Each heatmap is blended directly with the original image. The captions are generated from the strongest part of each map and state what region the map suggests the model used.

In [ ]:
results = [explain_one_image(index) for index in selected_indices]

for number, result in enumerate(results, start=1):
    true_name = label_names[result["true_class"]]
    predicted_name = label_names[result["predicted_class"]]
    status = "CORRECT" if result["correct"] else "MISCLASSIFIED"

    ig_caption = (
        "Integrated Gradients: strongest pixel evidence is "
        f"{heatmap_location(result['ig_heatmap'])}, "
        f"{attention_description(result['ig_heatmap'])}."
    )
    cam_caption = (
        "Grad-CAM: the broad predicted-class region is "
        f"{heatmap_location(result['cam_heatmap'])}, "
        f"{attention_description(result['cam_heatmap'])}."
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    axes[0].imshow(result["image"])
    axes[0].set_title(
        f"Input\nTrue: {true_name} | Predicted: {predicted_name}\n"
        f"Confidence: {result['confidence']:.3f}", fontsize=10
    )
    axes[1].imshow(result["ig_overlay"])
    axes[1].set_title(ig_caption, fontsize=9, wrap=True)
    axes[2].imshow(result["cam_overlay"])
    axes[2].set_title(cam_caption, fontsize=9, wrap=True)

    for ax in axes:
        ax.axis("off")

    fig.suptitle(
        f"Example {number}: {status} — test index {result['index']}",
        fontsize=13, y=1.02
    )
    plt.tight_layout()
    plt.show()

    print(
        f"Integrated Gradients completeness delta: {result['ig_delta']:.6f} | "
        f"Top-region IG/Grad-CAM overlap: {result['overlap']:.3f}"
    )

## 6. Comparison and interpretation

Integrated Gradients should appear more detailed because it works at the input-pixel level. Grad-CAM should appear smoother and wider because it starts from a lower-resolution deep feature map. To support the visual comparison, I also calculate the intersection-over-union of the strongest 20% of pixels in the two maps. This overlap number is only a spatial comparison; it is not proof that either explanation is correct.

In [ ]:
def agreement_word(overlap):
    if overlap >= 0.35:
        return "strongly agree"
    if overlap >= 0.18:
        return "partly agree"
    return "have low spatial agreement"

lines = ["### Per-image comparison"]
for number, result in enumerate(results, start=1):
    status = "correct" if result["correct"] else "misclassified"
    lines.append(
        f"- **Example {number} ({status}):** The two methods "
        f"{agreement_word(result['overlap'])} "
        f"(top-region IoU = {result['overlap']:.3f}). "
        f"Integrated Gradients is {attention_description(result['ig_heatmap'])}, "
        f"while Grad-CAM is {attention_description(result['cam_heatmap'])}."
    )

correct_results = [r for r in results if r["correct"]]
wrong_result = next(r for r in results if not r["correct"])
mean_correct_overlap = float(np.mean([r["overlap"] for r in correct_results]))
wrong_overlap = wrong_result["overlap"]

lines += [
    "",
    "### Overall interpretation",
    f"The three correct examples have an average IG/Grad-CAM overlap of **{mean_correct_overlap:.3f}**. "
    f"The misclassified example has an overlap of **{wrong_overlap:.3f}**.",
]

wrong_both_centered = (
    wrong_result["ig_center_ratio"] >= 0.58
    and wrong_result["cam_center_ratio"] >= 0.58
)
wrong_both_outer = (
    wrong_result["ig_center_ratio"] <= 0.40
    and wrong_result["cam_center_ratio"] <= 0.40
)

if wrong_both_outer:
    mistake_text = (
        "In the misclassified image, both explanations place much of their importance "
        "outside the central subject area. This suggests the model may have relied on "
        "background, edges, or another dataset artifact instead of the animal itself."
    )
elif wrong_both_centered and wrong_overlap >= 0.18:
    mistake_text = (
        "In the misclassified image, both explanations still focus mainly on the central "
        "subject area. This suggests the model looked at the animal but confused its visual "
        "features with the other class, rather than making the mistake only because of the background."
    )
else:
    mistake_text = (
        "In the misclassified image, the methods focus on different regions or split attention "
        "between the subject and surroundings. This disagreement suggests the prediction was "
        "not based on one clear, stable visual cue."
    )

lines += [
    mistake_text,
    "Where the two methods agree, there is stronger evidence that the highlighted region affected "
    "the prediction. Where they disagree, the difference is consistent with their resolution "
    "trade-off: Integrated Gradients identifies fine pixel evidence, while Grad-CAM identifies "
    "a broad deep-feature region.",
    "The overlays should still be inspected directly before making a final claim about whether a "
    "highlighted area is the animal, the background, or a specific feature such as the face, fur, ears, or body outline.",
]

display(Markdown("\n".join(lines)))

## Conclusion

The two explainers answer related but different questions. Integrated Gradients shows which input pixels contributed most strongly along the path from a black baseline to the real image. Grad-CAM shows which broad region of the last convolutional feature maps supported the predicted class. Agreement between them makes an interpretation more convincing, while disagreement or attention outside the animal can reveal uncertainty or a possible shortcut learned from the dataset.